# Ensemble Translation Inference — ByT5 + QLoRA Qwen + mBART-50

Loads three independently-trained translation checkpoints, generates a candidate translation from each, and picks a final answer per sentence using a **Minimum Bayes Risk (MBR) consensus orchestrator**.

**`EXECUTION_MODE`** (next cell): `"online"` installs packages, logs in to Hugging Face, and sets up MLflow tracking; pulls mBART-50 and Qwen from the Hub. `"offline"` skips all of that — every model loads from local Kaggle input paths, no network access.

**ByT5** is always local (Kaggle Model only, no Hub repo), regardless of mode.

Source text is used as-is from `train.csv`/`test.csv`, no normalization step.

Models are loaded, run, and fully unloaded one at a time to keep peak GPU memory at "one model" instead of three.

In [12]:
EXECUTION_MODE = "offline"  # "online" or "offline"
assert EXECUTION_MODE in ("online", "offline")
IS_ONLINE = EXECUTION_MODE == "online"
print(f"Running in {EXECUTION_MODE.upper()} mode")


Running in OFFLINE mode


In [13]:
SEED = 42

import random
import numpy as np
import torch
from transformers import set_seed as _hf_set_seed

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    _hf_set_seed(seed)

seed_everything()


In [14]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [15]:
import os
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()

# Local Kaggle input paths (offline mode) — dedicated weights account, safe to hardcode.
MBART_OFFLINE_PATH = "/kaggle/input/models/eeee13/akkadian-mbart/pytorch/subword-seq2seq-v1/1"
QWEN_BASE_OFFLINE_PATH = "/kaggle/input/models/eeee13/qwen2.5-7b-instruct/pytorch/base-v1/1"
QWEN_ADAPTER_OFFLINE_PATH = "/kaggle/input/models/eeee13/akkadika/pytorch/adapter-v1/1"
BYT5_REPO = "/kaggle/input/models/eeee13/akkadian-byt5/pytorch/byt5-akkadian-v4/1"

# Hub repo ids + credentials — kept in Secrets.
HF_TOKEN = _secrets.get_secret("HF_TOKEN") if IS_ONLINE else None
MLFLOW_USERNAME = _secrets.get_secret("MLFLOW_USERNAME") if IS_ONLINE else None
MLFLOW_PASSWORD = _secrets.get_secret("MLFLOW_PASSWORD") if IS_ONLINE else None
MBART_REPO = _secrets.get_secret("HF_REPO_MBART") if IS_ONLINE else None
QWEN_BASE_REPO = "Qwen/Qwen2.5-7B-Instruct" if IS_ONLINE else None
QWEN_ADAPTER_REPO = _secrets.get_secret("HF_REPO_QWEN_ADAPTER") if IS_ONLINE else None

if IS_ONLINE:
    for _name, _val in [("MBART_REPO", MBART_REPO), ("QWEN_ADAPTER_REPO", QWEN_ADAPTER_REPO)]:
        assert _val, f"{_name} is empty — check Add-ons > Secrets."
else:
    for _name, _val in [("MBART_OFFLINE_PATH", MBART_OFFLINE_PATH),
                         ("QWEN_BASE_OFFLINE_PATH", QWEN_BASE_OFFLINE_PATH),
                         ("QWEN_ADAPTER_OFFLINE_PATH", QWEN_ADAPTER_OFFLINE_PATH)]:
        assert _val, f"{_name} is empty — fill in the local Kaggle input path above."


In [16]:
import sys
import subprocess

if IS_ONLINE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "accelerate", "peft", "bitsandbytes", "sentencepiece", "mlflow"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dagshub", "--no-deps"], check=True)

    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
    else:
        print("WARNING: HF_TOKEN secret is empty — private Hub repo pulls will fail.")
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links=/kaggle/input/datasets/eeee13/bitsandbytes", "bitsandbytes"],
        check=True,
    )
    print("Offline mode: installed bitsandbytes from local wheel, skipping HF login.")


Offline mode: installed bitsandbytes from local wheel, skipping HF login.


## MLflow tracking

Logged once per model plus once for the ensemble, on the validation split:

- `val_bleu`, `val_chrf_pp` — corpus BLEU / chrF++ against the validation references
- `val_geo_mean_chrf_bleu` — geometric mean of the two
- `val_avg_logprob` — the mean beam-search sequence log-probability `generate()` returned, used here as a rough **stand-in for a validation loss** (it's not a true loss — these are frozen, already-trained checkpoints, there's no training loop here to compute one from — but it's the only per-sentence confidence signal available at inference time and moves in the same direction as one)

Each model gets its own nested MLflow run under a parent `ensemble-inference` run, so you can compare them side by side in the MLflow/DagsHub UI the same way you compare training runs.

In [17]:
if IS_ONLINE:
    import os
    import mlflow

    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

    MLFLOW_TRACKING_URI = _secrets.get_secret("MLFLOW_TRACKING_URI")
    assert MLFLOW_TRACKING_URI, "MLFLOW_TRACKING_URI is empty — check Add-ons > Secrets."

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("deep-past-initiative-machine-translation")
    print("MLflow tracking configured.")
else:
    print("Offline mode: skipping MLflow setup — validation metrics will still print to stdout.")


Offline mode: skipping MLflow setup — validation metrics will still print to stdout.


## Load the data once, up front — raw `train.csv` / `test.csv`, no normalization

`train.csv` doesn't ship with a `split` column, so a document-level train/val split is computed here (grouped by whichever id column is present, so a document's style/vocabulary can't leak between train and val) using the same seeded approach as the training notebook. Neither `val_texts` nor `test_texts` get any text transformation applied — this is intentionally the raw column content.

In [18]:
import random
import pandas as pd

COMP_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
VAL_FRACTION = 0.1

raw_train = pd.read_csv(f"{COMP_DIR}/train.csv")
raw_train = raw_train.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)

_id_col = next((c for c in ("text_id", "oare_id", "document_id") if c in raw_train.columns), None)
rng = random.Random(SEED)
if _id_col is not None:
    doc_ids = sorted(raw_train[_id_col].dropna().unique().tolist())
    rng.shuffle(doc_ids)
    n_val_docs = max(1, int(len(doc_ids) * VAL_FRACTION))
    val_ids = set(doc_ids[:n_val_docs])
    raw_train["split"] = raw_train[_id_col].apply(lambda x: "val" if x in val_ids else "train")
else:
    # no document-id column — falling back to a row-level split (risks train/val leakage)
    print(f"WARNING: no document-id column found among {list(raw_train.columns)} — "
          f"using a row-level random split instead of a document-level one.")
    idx = list(raw_train.index)
    rng.shuffle(idx)
    n_val = max(1, int(len(idx) * VAL_FRACTION))
    val_idx = set(idx[:n_val])
    raw_train["split"] = ["val" if i in val_idx else "train" for i in raw_train.index]

val_df = raw_train[raw_train["split"] == "val"].reset_index(drop=True)
val_texts = val_df["transliteration"].tolist()   # no normalization
val_refs = val_df["translation"].tolist()

test_df = pd.read_csv(f"{COMP_DIR}/test.csv")
test_texts = test_df["transliteration"].tolist()  # no normalization

print(f"val: {len(val_texts)} sentences, test: {len(test_texts)} sentences")


val: 156 sentences, test: 4 sentences


## Per-model load / generate / unload

Each model gets a `load_*()` function (returns whatever `generate_*()` needs) and a `generate_*()` function (batched, returns `(texts, confidences)` — confidences are the beam-search `sequences_scores` log-probs, used later to weight votes). `load_byt5()` always resolves to the local Kaggle path; `load_mbart()`/`load_qwen()` always resolve to Hub repo ids. Nothing here loads a model yet; the loop in the next cell does that one model at a time.

In [19]:
import gc
import math
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
NUM_BEAMS = 4
GEN_BATCH_SIZE = 8


def unload(ctx=None):
    if isinstance(ctx, dict):
        ctx.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _seq2seq_generate_batched(model, tokenizer, texts, forced_bos_token_id=None):
    all_texts, all_scores = [], []
    for i in tqdm(range(0, len(texts), GEN_BATCH_SIZE), leave=False):
        chunk = texts[i:i + GEN_BATCH_SIZE]
        inputs = tokenizer(
            chunk, return_tensors="pt", truncation=True,
            max_length=MAX_SOURCE_LENGTH, padding=True,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=NUM_BEAMS,
                return_dict_in_generate=True,   # output_scores removed
            )
        scores = (out.sequences_scores.tolist() if getattr(out, "sequences_scores", None) is not None
                  else [0.0] * len(out.sequences))
        decoded = tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
        all_texts.extend(d.strip() for d in decoded)
        all_scores.extend(scores)
    return all_texts, all_scores


# --- ByT5 (Kaggle-only checkpoint) --------------------------------------------
def load_byt5():
    tok = AutoTokenizer.from_pretrained(BYT5_REPO)
    model = AutoModelForSeq2SeqLM.from_pretrained(BYT5_REPO).to(DEVICE).eval()
    return {"model": model, "tokenizer": tok}


def generate_byt5(ctx, texts):
    return _seq2seq_generate_batched(ctx["model"], ctx["tokenizer"], texts)


# --- mBART-50 (Hub) -------------------------------------------------------------
def load_mbart():
    repo = MBART_REPO if IS_ONLINE else MBART_OFFLINE_PATH
    tok = AutoTokenizer.from_pretrained(repo)
    tok.src_lang = "en_XX"
    tok.tgt_lang = "en_XX"
    model = AutoModelForSeq2SeqLM.from_pretrained(repo).to(DEVICE).eval()
    forced_bos = tok.convert_tokens_to_ids
    return {"model": model, "tokenizer": tok, "forced_bos": forced_bos}


def generate_mbart(ctx, texts):
    return _seq2seq_generate_batched(ctx["model"], ctx["tokenizer"], texts, forced_bos_token_id=ctx["forced_bos"])


# --- QLoRA Qwen (Hub) ------------------------------------------------------------
QWEN_PROMPT_TEMPLATE = (
    "Translate the following Akkadian transliteration into English.\n"
    "Transliteration: {source}\n"
    "Translation:"
)


def load_qwen():
    base_repo = QWEN_BASE_REPO if IS_ONLINE else QWEN_BASE_OFFLINE_PATH
    adapter_repo = QWEN_ADAPTER_REPO if IS_ONLINE else QWEN_ADAPTER_OFFLINE_PATH

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(base_repo)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"  # required for batched generate() with a causal LM

    base = AutoModelForCausalLM.from_pretrained(
        base_repo, quantization_config=bnb_config, device_map="auto",
    )
    model = PeftModel.from_pretrained(base, adapter_repo).eval()
    return {"model": model, "tokenizer": tok}


def generate_qwen(ctx, texts):
    model, tok = ctx["model"], ctx["tokenizer"]
    all_texts, all_scores = [], []
    for i in tqdm(range(0, len(texts), GEN_BATCH_SIZE), leave=False):
        chunk = texts[i:i + GEN_BATCH_SIZE]
        prompts = [QWEN_PROMPT_TEMPLATE.format(source=t) for t in chunk]
        inputs = tok(
            prompts, return_tensors="pt", truncation=True,
            max_length=MAX_SOURCE_LENGTH, padding=True,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=NUM_BEAMS,
                return_dict_in_generate=True,   # output_scores removed
            )
        scores = (out.sequences_scores.tolist() if getattr(out, "sequences_scores", None) is not None
                  else [0.0] * len(out.sequences))
        gen_only = out.sequences[:, inputs["input_ids"].shape[1]:]  # strip the prompt back off
        decoded = tok.batch_decode(gen_only, skip_special_tokens=True)
        all_texts.extend(d.strip() for d in decoded)
        all_scores.extend(scores)
    return all_texts, all_scores


MODEL_SPECS = {
    "byt5": {"load": load_byt5, "generate": generate_byt5},
    "mbart": {"load": load_mbart, "generate": generate_mbart},
    "qwen": {"load": load_qwen, "generate": generate_qwen},
}


## Sequential run — one model in memory at a time

For each model: load it, translate the *entire* val set and the *entire* test set, stash the results, then unload it before moving to the next. `predictions[model_name]["val"|"test"]` holds `(texts, confidences)` afterwards; no model weights are kept around once this cell finishes.

In [20]:
seed_everything()  # re-seed before generation

predictions = {}

for name, spec in MODEL_SPECS.items():
    print(f"\n=== Loading {name} ===")
    ctx = spec["load"]()

    print(f"Translating validation set with {name}...")
    val_texts_out, val_scores_out = spec["generate"](ctx, val_texts)

    print(f"Translating test set with {name}...")
    test_texts_out, test_scores_out = spec["generate"](ctx, test_texts)

    predictions[name] = {
        "val": (val_texts_out, val_scores_out),
        "test": (test_texts_out, test_scores_out),
    }

    print(f"Unloading {name}...")
    del val_texts_out, val_scores_out, test_texts_out, test_scores_out
    unload(ctx)
    del ctx
    gc.collect(); torch.cuda.empty_cache()
    print(torch.cuda.memory_allocated() / 1e9, "GB still allocated")

print("\nAll three models translated and unloaded. Peak GPU memory was one model at a time.")



=== Loading byt5 ===


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Translating validation set with byt5...


KeyboardInterrupt: 

## Orchestration algorithm — Minimum Bayes Risk consensus

For each sentence, all three models proposed a candidate (already generated and stored above). Instead of a fixed rule ("always trust mBART"), the orchestrator scores each candidate by its **chrF similarity to the other two candidates**, weighted by each model's decoding confidence (and an optional tunable prior per model). The candidate that best "agrees" with the others wins.

This works because the three models are structurally diverse (byte-level seq2seq, LoRA-tuned decoder-only LM, denoising seq2seq) — they're unlikely to make the *same* mistake by chance, so agreement is a reasonable proxy for correctness even without a reference translation at inference time. It degrades gracefully: if one model fails to produce output, MBR falls back to whichever of the remaining candidates agrees most; if only one model produces output, it's used as-is.

Tune `MODEL_PRIORS` from the validation results in the next section if one model turns out systematically stronger.

In [ ]:
from collections import Counter

def _char_ngrams(s, n):
    s = s.replace(" ", "")
    return Counter(s[i:i + n] for i in range(len(s) - n + 1)) if len(s) >= n else Counter()


def sentence_chrf(a, b, max_n=6, beta=2):
    """Sentence-level chrF (character n-gram F-score), used as the MBR utility function."""
    if not a or not b:
        return 0.0
    ps, rs = [], []
    for n in range(1, max_n + 1):
        ca, cb = _char_ngrams(a, n), _char_ngrams(b, n)
        overlap = sum((ca & cb).values())
        ps.append(overlap / max(sum(ca.values()), 1))
        rs.append(overlap / max(sum(cb.values()), 1))
    p, r = sum(ps) / max_n, sum(rs) / max_n
    if p + r == 0:
        return 0.0
    beta2 = beta ** 2
    return (1 + beta2) * p * r / (beta2 * p + r)


# Relative trust per model — start at 1.0 each; adjust after looking at per-model
# validation chrF/BLEU below (e.g. downweight a model that's consistently worse).
MODEL_PRIORS = {"byt5": 1.0, "mbart": 1.0, "qwen": 1.0}


def mbr_orchestrate(candidates):
    """
    candidates: list of {"model": str, "text": str, "conf": float}  (conf = generate()'s
    sequence log-prob, <= 0; more negative = less confident)

    Returns (winner_dict, all_candidates_with_utility) — winner_dict has an extra
    "utility" key so you can inspect how confident the consensus was.
    """
    valid = [c for c in candidates if c["text"]]
    if not valid:
        return {"model": None, "text": "", "utility": 0.0}, []
    if len(valid) == 1:
        only = {**valid[0], "utility": 1.0}
        return only, [only]

    weights = []
    for c in valid:
        prior = MODEL_PRIORS.get(c["model"], 1.0)
        conf = math.exp(c["conf"]) if c["conf"] <= 0 else c["conf"]  # log-prob -> (0, 1]
        weights.append(prior * conf)
    total_w = sum(weights) or 1.0
    weights = [w / total_w for w in weights]

    scored = []
    for i, ci in enumerate(valid):
        utility = sum(
            weights[j] * sentence_chrf(ci["text"], cj["text"])
            for j, cj in enumerate(valid) if j != i
        )
        scored.append({**ci, "utility": utility})

    winner = max(scored, key=lambda c: c["utility"])
    return winner, scored


def orchestrate_all(split):
    """split: 'val' or 'test'. Reads from the `predictions` dict filled in by the sequential run."""
    n = len(predictions["byt5"][split][0])
    finals, log = [], []
    for i in range(n):
        candidates = [
            {"model": name, "text": predictions[name][split][0][i], "conf": predictions[name][split][1][i]}
            for name in MODEL_SPECS
        ]
        winner, scored = mbr_orchestrate(candidates)
        finals.append(winner["text"])
        log.append(scored)
    return finals, log


## Validation — ensemble vs. each individual model

Pure-Python corpus BLEU/chrF++ (no `evaluate`/`sacrebleu` dependency). Also logs `val_bleu`, `val_chrf_pp`, `val_geo_mean_chrf_bleu` (and a `val_avg_logprob` confidence proxy) to MLflow for each model and for the ensemble, under a shared `ensemble-inference` parent run — only when `EXECUTION_MODE == "online"`; metrics still print to stdout either way.

In [ ]:
import numpy as np

def _ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(preds, refs, max_n=4):
    clipped, total = [0] * max_n, [0] * max_n
    pred_len, ref_len = 0, 0
    for pred, ref in zip(preds, refs):
        p_tok, r_tok = pred.split(), ref.split()
        pred_len += len(p_tok); ref_len += len(r_tok)
        for n in range(1, max_n + 1):
            pc, rc = _ngram_counts(p_tok, n), _ngram_counts(r_tok, n)
            clipped[n - 1] += sum(min(c, rc[g]) for g, c in pc.items())
            total[n - 1] += max(sum(pc.values()), 0)
    precisions = [clipped[n] / total[n] if total[n] > 0 else 0.0 for n in range(max_n)]
    geo = 0.0 if min(precisions) == 0 else math.exp(sum(math.log(p) for p in precisions) / max_n)
    bp = 1.0 if pred_len > ref_len else math.exp(1 - ref_len / max(pred_len, 1))
    return 100.0 * bp * geo


def corpus_chrf(preds, refs, n=6, beta=2):
    tp, tt, rc_, rt = [0] * n, [0] * n, [0] * n, [0] * n
    for pred, ref in zip(preds, refs):
        pc, rc = pred.replace(" ", ""), ref.replace(" ", "")
        for k in range(1, n + 1):
            pcs, rcs = _ngram_counts(pc, k), _ngram_counts(rc, k)
            overlap = sum((pcs & rcs).values())
            tp[k - 1] += overlap; tt[k - 1] += max(sum(pcs.values()), 0)
            rc_[k - 1] += overlap; rt[k - 1] += max(sum(rcs.values()), 0)
    precisions = [tp[k] / tt[k] if tt[k] > 0 else 0.0 for k in range(n)]
    recalls = [rc_[k] / rt[k] if rt[k] > 0 else 0.0 for k in range(n)]
    p, r = sum(precisions) / n, sum(recalls) / n
    if p + r == 0:
        return 0.0
    beta2 = beta ** 2
    return 100.0 * (1 + beta2) * p * r / (beta2 * p + r)


def geometric_mean(a, b):
    """Geometric mean of two non-negative metrics (chrF++ and BLEU, both on a 0-100 scale)."""
    a, b = max(a, 0.0), max(b, 0.0)
    return math.sqrt(a * b)


In [ ]:
ensemble_val, ensemble_val_log = orchestrate_all("val")

def _avg_logprob(scores):
    scores = [s for s in scores if s is not None]
    return sum(scores) / len(scores) if scores else None

print(f"{'system':10s}  {'BLEU':>6s}  {'chrF++':>7s}  {'geo_mean':>8s}")

mlflow_parent_run = mlflow.start_run(run_name="ensemble-inference") if IS_ONLINE else None
parent_metrics = {}

per_model_metrics = {}
for name in MODEL_SPECS:
    preds, scores = predictions[name]["val"]
    bleu = corpus_bleu(preds, val_refs)
    chrf_pp = corpus_chrf(preds, val_refs)
    geo_mean = geometric_mean(chrf_pp, bleu)
    per_model_metrics[name] = {"val_bleu": bleu, "val_chrf_pp": chrf_pp, "val_geo_mean_chrf_bleu": geo_mean}
    print(f"{name:10s}  {bleu:6.2f}  {chrf_pp:7.2f}  {geo_mean:8.2f}")

    log_dict = dict(per_model_metrics[name])
    avg_lp = _avg_logprob(scores)
    if avg_lp is not None:
        log_dict["val_avg_logprob"] = avg_lp  # rough confidence proxy, not a true loss

    # Log every metric on the per-model nested run AND, prefixed, on the parent run —
    # so metrics show up whichever run entry gets checked on DagsHub.
    if IS_ONLINE:
        with mlflow.start_run(run_name=f"val-{name}", nested=True):
            mlflow.set_tag("model", name)
            mlflow.log_metrics(log_dict)
    for k, v in log_dict.items():
        parent_metrics[f"{name}_{k}"] = v

ensemble_bleu = corpus_bleu(ensemble_val, val_refs)
ensemble_chrf_pp = corpus_chrf(ensemble_val, val_refs)
ensemble_geo_mean = geometric_mean(ensemble_chrf_pp, ensemble_bleu)
print(f"{'ensemble':10s}  {ensemble_bleu:6.2f}  {ensemble_chrf_pp:7.2f}  {ensemble_geo_mean:8.2f}")

ensemble_metrics = {
    "val_bleu": ensemble_bleu,
    "val_chrf_pp": ensemble_chrf_pp,
    "val_geo_mean_chrf_bleu": ensemble_geo_mean,
}
if IS_ONLINE:
    with mlflow.start_run(run_name="val-ensemble", nested=True):
        mlflow.set_tag("model", "ensemble")
        mlflow.log_metrics(ensemble_metrics)
for k, v in ensemble_metrics.items():
    parent_metrics[f"ensemble_{k}"] = v

winner_counts = Counter(max(row, key=lambda c: c['utility'])['model'] for row in ensemble_val_log)
print("\nWinning model per sentence:", dict(winner_counts))

if IS_ONLINE:
    # Log the full combined set (every model + ensemble) directly on the parent run too.
    mlflow.log_metrics(parent_metrics)
    mlflow.log_metrics({f"winner_count_{k}": v for k, v in winner_counts.items()})
    mlflow.end_run()  # closes mlflow_parent_run


## Test-set inference (submission.csv)

Also writes `ensemble_candidates_debug.csv` with every model's raw candidate + which one the orchestrator picked, for auditing. No model loading or network access happens here — it's all reading from `predictions`, already populated by the sequential run above.

In [ ]:
final_translations, test_log = orchestrate_all("test")

submission = pd.DataFrame({"id": test_df["id"], "translation": final_translations})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

debug_rows = []
for row_id, scored_row in zip(test_df["id"], test_log):
    entry = {"id": row_id}
    for c in scored_row:
        entry[f"{c['model']}_text"] = c["text"]
        entry[f"{c['model']}_utility"] = c["utility"]
    entry["chosen_model"] = max(scored_row, key=lambda c: c["utility"])["model"]
    debug_rows.append(entry)
pd.DataFrame(debug_rows).to_csv("ensemble_candidates_debug.csv", index=False)
print("Saved ensemble_candidates_debug.csv")
